In [ ]:
import seaborn as sns
from sklearn.model_selection import KFold
import matplotlib.pyplot as plt

from scipy import stats
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from formulaic import model_matrix

In [ ]:
#import os
#print(pwd)
#os.chdir('./notebooks')

# Limpieza de Datos

En Python, las funciones y paquetes para trabajar con datos no están pensadas para poder hacerlas "sin saber nada de Python". Al contrario, muchas operaciones sencillas de DataFrames requieren conocer técnicas de programación en Python. Vamos a ver algunas en esta clase, principalmente estas cuatro:

## 1. Diccionarios
## 2. Encadenamiento de métodos
## 3. Funciones lambda
## 4. Definición de listas por comprensión

**Ejercicio 1.** En el dataset de vinos de la última clase, renombrar la columna "od280/od315_of_diluted_wines" a "od280_od315_of_diluted_wines".

In [ ]:
from sklearn.datasets import load_wine
wine = load_wine(as_frame=True)
wineData = wine.data
wineData.head()

Hay esencialmente dos formas de renombrar columnas:
1. Utilizar `rename()`
2. Modificar la lista de nombres de columnas `dataframe.columns`

Comenzamos con `rename()`.

In [ ]:
pd.DataFrame.rename??

# O help(pd.DataFrame.rename)
# O wineData.rename??

Vemos que podemos renombrar columnas utilizando `columns: dict-like or function`

## 1. Diccionarios
Los diccionarios se utilizan para almacenar valores de datos en pares clave:valor.

Un diccionario es una colección ordenada, modificable y que no permite duplicados.

In [ ]:
# Ejemplo
dict = { "hola" : "Expresión de saludo utilizada entre dos o más personas de trato familiar", "cinco" : 5, "dias" : ["lunes", "martes", "miercoles"]}

In [ ]:
dict["cinco"]

In [ ]:
# Ejemplo
thisdict = {
  "brand": "Ford",
  "model": "Mustang",
  "year": 1964
}
print(thisdict["brand"])

Para renombrar una columna, usamos rename(columns = _DICT_) y un diccionario con "nombre viejo" como claves y "nombre nuevo" como valores.

In [ ]:
wineData.rename(columns = {"od280/od315_of_diluted_wines" : "od280_od315_of_diluted_wines"})  # No es necesario renombrar todas las columnas, podemos renombrar las que querramos.

In [ ]:
# Cambió algo?
wineData

In [ ]:
# Podemos guardar los cambios en el mismo DataFrame o en uno nuevo
wineClean = wineData.rename(columns = {"od280/od315_of_diluted_wines" : "od280_od315_of_diluted_wines"})
wineClean.head()

**Nota:** también existe la opción "inplace = True" pero no está recomendada (por ejemplo, no permite encadenar comandos).

### `rename` con funciones

También podemos usar funciones para renombrar columnas (Python permite pasar funciones como parámetros).

**Ejemplo:** Pasar todos los nombres de columnas a mayúsculas.

In [ ]:
# Para pasar un string a mayusculas utilizamo str.upper
str.upper("hola")

¿Cómo lo usarian para renombrar nuestras variables?

In [ ]:
# Así?
#datos.rename(columns = str.upper)
# o así?
#datos.rename(columns = str.upper())

<details>
<summary>Click para ver la respuesta</summary>

Le pasamos el nombre de la función, por eso va sin los paréntesis. 

Si la pasamos con paréntesis le estaríamos pasando el resultado de evaluar la función sin parámetros.
</details>


### Ejercicio

Definiendo tu propia función, agregar en todos los nombres de columna "_2026" al final del nombre.

## 2. Encadenar métodos

### 2.a Datos faltantes

**Ejercicio 2.** En la base de dato de pingüinos, eliminar las filas con datos faltantes, y renumerar los índices comenzando en 1.

In [ ]:
penguins = sns.load_dataset("penguins")
penguins.head()

## Primero: ¿qué son los datos faltantes?

¿Qué significa un valor faltante? Es un dato que por distintos motivos no fue ingresado (por ejemplo si juntamos datos de distintas fuentes, en alguna de las fuentes esa información puede no estar disponible).

Los datos faltantes son algo típico de cualquier pipeline de análisis de datos. En general los paquetes estadísticos (pandas, dplyr, etc) tratan de facilitar el manejo de estos tipos de datos. En Pandas, se utiliza la expresión NaN (Not a number) o NA que viene del mundo R (Not available)

In [ ]:
# Vamos a trabajar con una base de eficiencia del sueño
datos = pd.read_csv("Sleep_efficiency_cleaning.csv")
datos   

Para poder acceder a los datos faltantes, podemos utilizar la función `isna()`

¿Cómo la usamos? ¿Qué nos devuelve? 

In [ ]:
datos.isna()

¿Cómo podemos transformar toda esta información para que sea más legible?

In [ ]:
# Una forma bien simple sería usando 
datos.isna().sum()

In [ ]:
# De ahi sabemos que Despertares, Consumo de cafeina, Consumo de alcohol y Frecuencia de ejercicio tienen datos faltantes

In [ ]:
# Esto nos da lo mismo (null es comun en SQL, NaN se usa en NumPy)
penguins.isnull().sum()

¿Qué podríamos hacer con todos estos datos faltantes?

In [ ]:
# Una posibilidad es usar dropna() 
# ¿Qué les parece que hace esta función? 
# ¿Cuántas filas teniamos antes y cuántas tenemos ahora? ¿Cuáles son las ventajas y desventajas de esto?

datos.dropna()

In [ ]:
# Otra posibilidad es eliminar las columnas con datos faltantes, de forma de no perder participantes. ¿Cuáles son las ventajas y desventajas de esto?
datos.dropna(axis="columns")

Otra opción es hacer una combinación de estas dos posibilidades. Por ejemplo, eliminar las columnas con muchos datos faltantes (e.g. Awakenings y Caffeine compsumption) y recien luego de eso eliminar las filas que tengan algun dato perdido. 

De esa forma no pierdo tantas filas ni tantas columnas.

In [ ]:
datos1 = datos.drop(columns=["Awakenings", "Caffeine consumption"])
datos2 = datos1.dropna()
datos2

# En un rato vamos a ver que hay formas más eficientes y ordenadas de encadenar comandos como estos.

También podríamos elegir no eliminar los casos, sino reemplazar los datos faltantes con un valor (e.g. cero). A esto se lo llama "imputación de datos faltantes".

Lo podemos hacer con fillna()

In [ ]:
datos.fillna(0)

In [ ]:
# Chequeamos
filas_con_nan = datos.isna().any(axis=1)
datos[filas_con_nan]

In [ ]:
datos0 = datos.fillna(0)
datos0[filas_con_nan]

¿Qué opinamos de esto? ¿Se puede usar en todos los casos? ¿Qué otras formas se te ocurren?

In [ ]:
# Una forma super sencilla (pero no infalible) es reemplazar el dato perdido por la media o la mediana poblacional 
# (Capaz la mediana es mejor porque te va a buscar un dato existente)

# Ahora cual usamos??
#datos.fillna(mean)
#datos.fillna(datos.mean())

#Veamos el help... 
datos.fillna??

In [ ]:
# Tenemos que pasarle un vector de valores para usar en cada columna, el vector lo generamos asi:
datos.mean(numeric_only=True)   # No podemos tomar media de variables no-numericas

Podemos resolverlo utilizando funciones que ya vimos, modificando la variable index.

In [ ]:
# Esto va a funcionar?? fillna solo actúa en columnas numéricas??
datosMedia = datos.fillna(datos.mean(numeric_only=True))
datosMedia[filas_con_nan]

En pocas palabras:
- Completamos con 0 si pensamos que no figura el dato porque no hay o no corresponde. Por ejemplo, en información de alimentos y bebidas, el porcentaje de alcohol puede figurar como NA.
- Completamos con el promedio si pensamos que se perdió el dato y queremos afectar lo menos posible los modelos (o una técnica más avanzada sería predecir ese valor usando otras variables).

Cómo completarías estos datos faltantes?
- Metros cuadrados de un departamento.
- Cantidad de votos recibidos por un partido en la elección anterior.
- Cantidad de ventas realizadas de cierto producto en cierto día.
- Altura de un paciente.

### Ejercicio
1. Examinar el dataset de eficiencia MPG y hacer algo con los datos faltantes. ¿Qué harían si no queremos eliminar filas?
2. Examinar el dataset de pingüinos y hacer algo con los datos faltantes.

In [ ]:
penguins = sns.load_dataset("penguins")
mpg = sns.load_dataset("mpg")

### 2.b Eliminar datos duplicados

En muchos casos puede suceder que haya datos duplicados en nuestro dataset. ¿Se les ocurren ejemplos?

Para identificarlos, podemos usar `duplicated()`. ¿Qué está haciendo esta función?

In [ ]:
datos.duplicated()

In [ ]:
# Cuántos datos duplicados hay?
datos.duplicated().sum()

In [ ]:
# Es la misma fila repetida muchas veces o varias filas duplicadas?
# Podemos usar el viejo y querido value_counts()...
datos.value_counts()

Para eliminarlos, podríamos usar `drop_duplicates()`

In [ ]:
# O vemos las filas
datos[datos.duplicated()]

# O keep = False para ver todas las filas duplicadas, todas las veces que aparecen
datos[datos.duplicated(keep = False)] 

In [ ]:
# Eliminamos duplicados
datos.drop_duplicates()

¿Qué hace esta función? ¿Qué valores se queda y cuáles elimina?

Por defecto, se queda con el primer caso, pero podemos indicarle que se quede con el último por ej:

In [ ]:
datos.drop_duplicates(keep = "last")

**Pregunta:** Si tuvieramos sujetos duplicados, es decir, completaron dos veces la misma encuesta, pero respondieron distinto, cómo lo podríamos saber?

### 2.c Resetear el index

Volvamos a los pingüinos.
Una vez que eliminamos filas por el motivo que sea, nos van a quedar índices salteados. En muchos casos no es importante conservar los índices originales, y es conveniente numerar todo de 0 a N-1.

In [ ]:
penguinsClean = penguins.dropna()
penguinsClean

In [ ]:
# Reseteamos los índices 
penguinsClean = penguinsClean.reset_index()
penguinsClean   # Vemos que creó una nueva columna con los índices viejos

In [ ]:
# De esta forma solo nos quedamos con lo síndices nuevos.
penguinsClean = penguinsClean.reset_index(drop = True)
penguinsClean

In [ ]:
# No funciono??
# Es que tenemos que empezar todo de nuevo!

penguinsClean = penguins.dropna()
penguinsClean = penguinsClean.reset_index(drop = True)
penguinsClean

## Encadenar todos estos métodos

Esta forma de ir modificando un dataset no es muy práctica.

Por ejemplo si ejecutamos varias veces el mismo comando, obtendremos cada vez algo distinto.

Una solución es encadenar métodos para hacer el código más legible y fácil de reusar. Y más eficiente también.

In [ ]:
penguinsClean = penguins.dropna().reset_index()
penguinsClean.head()

In [ ]:
# Si queremos eliminar la columna de índices viejos
penguinsClean = penguins.dropna().reset_index(drop = True)
penguinsClean.head()

In [ ]:
# Ahora sumamos 1 a todos los indices
penguinsClean.index += 1
penguinsClean.head()

Podemos encadenar estas operaciones?

In [ ]:
penguinsClean = penguins.dropna().reset_index(drop = True).index += 1

In [ ]:
# Eso dio error. Y así?
penguins.dropna().reset_index(drop = True).index += 1
penguins.head()

No sabemos dónde lo guardó...

Para encadenar esta operación vamos a utilizar `.rename(index = dict o function)`.

Podemos pasarle también un diccionario o una función.

### Ejercicio 
1. Cambiar el índice 0 a "Cero".
2. Tomas la raíz cuadradada de los índices (del dataset penguinsClean original sin "Cero").

In [ ]:
penguinsClean = penguins.dropna().reset_index(drop = True)
penguinsClean.head()

### Ejemplo: sumar uno a todos los índices

Para sumar uno a todos los índices no podemos hacer penguinsClean.rename(index = + 1).

Podemos definir primero nuestra propia función.

In [ ]:
def sumaUno(x):
    return(???)

penguinsClean.rename(index = ???)

## 3. Funciones lambda
Resulta engorroso definir una función para usarla solamente dentro del rename. 

Las **funciones lambda** nos permiten definir funciones anónimas "al vuelo".

La sintaxis es

`lambda arguments : expression`

In [ ]:
# Ejemplo
x = lambda a : a + 10
print(x(5))

In [ ]:
# Ejemplo
x = lambda a, b : a * b
print(x(5, 6))

In [ ]:
# Ejemplo
x = lambda a : a + 1
print(x(5))

In [ ]:
# Ahora sí!
penguinsClean.rename(index = lambda a : a + 1)

Esta operación la podemos encadenar con las anteriores:

In [ ]:
penguinsClean = penguins.dropna().reset_index(drop = True).rename(index = lambda a : a + 1)
penguinsClean.head()

In [ ]:
# O si queremos cada metodo en una linea nueva
penguinsClean = (
    penguins.dropna()
    .reset_index(drop = True)
    .rename(index = lambda a : a + 1)
)
penguinsClean.head()

### Ejercicio
Utilizando funciones lambda:
1. Encadenar un método más para agregar "_orig" a los nombres de todas las columnas.
2. Encadenar un método más para reemplazar todos _ por - en los nombres de columnas.

In [ ]:
# Sugerencia 1
"hola" + "chau"

In [ ]:
# Sugerencia 2
str.replace("abcndjaA", "a", "oo")